# Gold Company Hiring Mart

**Purpose**: Dashboard-ready company-level hiring activity mart for Superset visualization.

**Target Table**: `workspace.gold.gold_company_hiring`

**Grain**: One row per date per company per sector

**Key Metrics**:
- Active job counts
- New job postings
- Job closures
- Net hiring change
- Unique roles offered
- Hiring velocity indicators

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_company`
- `workspace.warehouse.dim_sector`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_company_hiring
USING DELTA
COMMENT 'Company-level hiring activity mart for dashboard consumption'
AS

WITH base_metrics AS (
  SELECT
    f.posting_date_sk AS company_date_sk,
    f.company_sk,
    f.sector_sk,
    
    -- MEASURES: Job counts
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN f.is_update = TRUE THEN 1 END) AS updated_jobs,
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    
    -- DERIVED MEASURES
    COUNT(CASE WHEN f.is_new_job = TRUE THEN 1 END) - 
    COUNT(CASE WHEN f.is_soft_delete = TRUE THEN 1 END) AS net_change,
    
    COUNT(DISTINCT f.role_sk) AS unique_roles,
    COUNT(DISTINCT f.location_sk) AS unique_locations,
    
    -- Support metrics
    COUNT(*) AS total_events
    
  FROM workspace.warehouse.fact_job_postings f
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
  GROUP BY f.posting_date_sk, f.company_sk, f.sector_sk
),

temporal_metrics AS (
  SELECT
    bm.*,
    
    -- TEMPORAL METRICS: Period-over-period comparisons
    LAG(bm.active_jobs, 1) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk 
      ORDER BY bm.company_date_sk
    ) AS prev_day_active_jobs,
    
    LAG(bm.active_jobs, 7) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk 
      ORDER BY bm.company_date_sk
    ) AS prev_week_active_jobs,
    
    LAG(bm.active_jobs, 30) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk 
      ORDER BY bm.company_date_sk
    ) AS prev_month_active_jobs,
    
    -- Week-to-date cumulative
    SUM(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk, 
        YEAR(TO_DATE(CAST(bm.company_date_sk AS STRING), 'yyyyMMdd')),
        WEEKOFYEAR(TO_DATE(CAST(bm.company_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY bm.company_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wtd_new_jobs,
    
    -- Month-to-date cumulative
    SUM(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk,
        YEAR(TO_DATE(CAST(bm.company_date_sk AS STRING), 'yyyyMMdd')),
        MONTH(TO_DATE(CAST(bm.company_date_sk AS STRING), 'yyyyMMdd'))
      ORDER BY bm.company_date_sk
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS mtd_new_jobs,
    
    -- 7-day rolling average
    AVG(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.company_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg_new_jobs,
    
    -- 30-day rolling average
    AVG(bm.new_jobs) OVER (
      PARTITION BY bm.company_sk, bm.sector_sk
      ORDER BY bm.company_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_avg_new_jobs
    
  FROM base_metrics bm
),

final_metrics AS (
  SELECT
    -- DIMENSIONS
    tm.company_date_sk,
    tm.company_sk,
    tm.sector_sk,
    
    -- MEASURES
    tm.active_jobs,
    tm.new_jobs,
    tm.updated_jobs,
    tm.closed_jobs,
    tm.net_change,
    tm.unique_roles,
    tm.unique_locations,
    tm.total_events,
    
    -- TEMPORAL METRICS
    tm.prev_day_active_jobs,
    tm.prev_week_active_jobs,
    tm.prev_month_active_jobs,
    tm.wtd_new_jobs,
    tm.mtd_new_jobs,
    tm.rolling_7day_avg_new_jobs,
    tm.rolling_30day_avg_new_jobs,
    
    -- DERIVED METRICS: Percent changes
    CASE 
      WHEN tm.prev_day_active_jobs > 0 
      THEN CAST((tm.active_jobs - tm.prev_day_active_jobs) AS DECIMAL(10,4)) / tm.prev_day_active_jobs
      ELSE NULL 
    END AS pct_change_vs_prev_day,
    
    CASE 
      WHEN tm.prev_week_active_jobs > 0 
      THEN CAST((tm.active_jobs - tm.prev_week_active_jobs) AS DECIMAL(10,4)) / tm.prev_week_active_jobs
      ELSE NULL 
    END AS pct_change_vs_prev_week,
    
    CASE 
      WHEN tm.prev_month_active_jobs > 0 
      THEN CAST((tm.active_jobs - tm.prev_month_active_jobs) AS DECIMAL(10,4)) / tm.prev_month_active_jobs
      ELSE NULL 
    END AS pct_change_vs_prev_month,
    
    -- Job churn rate
    CASE 
      WHEN tm.prev_day_active_jobs > 0 
      THEN CAST(tm.closed_jobs AS DECIMAL(10,4)) / tm.prev_day_active_jobs
      ELSE NULL 
    END AS job_churn_rate,
    
    -- Hiring velocity (new jobs / active jobs)
    CASE 
      WHEN tm.active_jobs > 0 
      THEN CAST(tm.new_jobs AS DECIMAL(10,4)) / tm.active_jobs
      ELSE NULL 
    END AS hiring_velocity,
    
    -- METADATA
    CURRENT_TIMESTAMP() AS created_at,
    UUID() AS batch_id
    
  FROM temporal_metrics tm
)

SELECT * FROM final_metrics
ORDER BY company_date_sk DESC, company_sk, sector_sk;

In [0]:
%sql
-- Validation: Check row counts and date ranges
SELECT
  'Overall Stats' AS metric_type,
  COUNT(*) AS row_count,
  COUNT(DISTINCT company_sk) AS unique_companies,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  MIN(company_date_sk) AS min_date_sk,
  MAX(company_date_sk) AS max_date_sk,
  ROUND(AVG(active_jobs), 2) AS avg_active_jobs,
  ROUND(AVG(new_jobs), 2) AS avg_new_jobs
FROM workspace.gold.gold_company_hiring

UNION ALL

-- Top 10 companies by total active jobs
SELECT
  'Top Company: ' || c.canonical_company_name AS metric_type,
  SUM(gch.active_jobs) AS row_count,
  NULL AS unique_companies,
  NULL AS unique_sectors,
  NULL AS min_date_sk,
  NULL AS max_date_sk,
  NULL AS avg_active_jobs,
  NULL AS avg_new_jobs
FROM workspace.gold.gold_company_hiring gch
JOIN workspace.warehouse.dim_company c ON gch.company_sk = c.company_sk
GROUP BY c.canonical_company_name
ORDER BY row_count DESC
LIMIT 10;